In [ ]:
!pip install vaderSentiment -q

import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from datetime import timezone

try:
    from google.colab import files
    print('Upload your two CSV files:')
    print('  - Bumble_decline data - Combined correct dates.csv')
    print('  - Bumble_decline_data_-_Kaggle_googleplay_dataset.csv')
    uploaded = files.upload()
    print(f'\nUploaded: {list(uploaded.keys())}')
except ImportError:
    print('Not in Colab — reading from local directory.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.1 MB/s eta 0:00:00


KeyboardInterrupt: 

In [ ]:
# ---------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------

INPUT_COMBINED = 'Bumble_decline data - Combined correct dates.csv'
INPUT_KAGGLE   = 'Bumble_decline data - Kaggle googleplay dataset.csv'
OUTPUT_FILE    = 'bumble_combined_full.csv'

# Final column order enforced across all sources
FINAL_COLUMNS = [
    'source',
    'original_date',
    'comment_date',
    'text',
    'comment',
    'subreddit',
    'post_title',
    'post_score',
    'comment_score',
    'rating',
    'thumbs_up',
    'vader_compound',
    'vader_positive',
    'vader_negative',
    'vader_neutral',
    'sentiment_label',
    'category_1',
    'category_2',
    'category_3',
    'dimension_1',
    'dimension_2',
]

print('Configuration loaded.')

In [ ]:
analyzer = SentimentIntensityAnalyzer()

def score_vader(text):
    """Run VADER and return scores + label."""
    if not isinstance(text, str) or len(text.strip()) == 0:
        return pd.Series([None, None, None, None, None])
    scores   = analyzer.polarity_scores(text)
    compound = scores['compound']
    label    = 'positive' if compound >= 0.05 else 'negative' if compound <= -0.05 else 'neutral'
    return pd.Series([
        round(compound, 4),
        round(scores['pos'], 4),
        round(scores['neg'], 4),
        round(scores['neu'], 4),
        label
    ])

print('Helper functions ready.')

In [ ]:
# ---------------------------------------------------------------
# LOAD EXISTING COMBINED FILE
# ---------------------------------------------------------------
df_combined = pd.read_csv(INPUT_COMBINED)

# Rename google_play source to google_play_scraped to distinguish from Kaggle
df_combined['source'] = df_combined['source'].replace('google_play', 'google_play_scraped')

# Add empty category/dimension columns if not present
for col in ['category_1', 'category_2', 'category_3', 'dimension_1', 'dimension_2']:
    if col not in df_combined.columns:
        df_combined[col] = None

print(f'Combined file loaded: {len(df_combined)} rows')
print(f'Sources: {df_combined["source"].value_counts().to_dict()}')
print(f'Columns: {list(df_combined.columns)}')

In [ ]:
# ---------------------------------------------------------------
# LOAD AND STANDARDISE KAGGLE DATASET
# ---------------------------------------------------------------
df_kaggle = pd.read_csv(INPUT_KAGGLE)
print(f'Kaggle file loaded: {len(df_kaggle)} rows')
print(f'Columns: {list(df_kaggle.columns)}')

# Filter to reviews with actual content
df_kaggle = df_kaggle[df_kaggle['content'].notna()]
df_kaggle = df_kaggle[df_kaggle['content'].str.strip().str.len() >= 10]
print(f'After filtering short/empty reviews: {len(df_kaggle)} rows')

# Parse date
df_kaggle['original_date'] = pd.to_datetime(
    df_kaggle['at'], errors='coerce'
).dt.strftime('%Y-%m-%d')

# Score VADER
print('Scoring VADER sentiment (this may take a minute)...')
vader_scores = df_kaggle['content'].apply(score_vader)
vader_scores.columns = ['vader_compound', 'vader_positive',
                        'vader_negative', 'vader_neutral', 'sentiment_label']

# Build standardised dataframe
df_kaggle_std = pd.DataFrame({
    'source':          'google_play_kaggle',
    'original_date':   df_kaggle['original_date'],
    'comment_date':    None,
    'text':            df_kaggle['content'],
    'comment':         df_kaggle['content'],
    'subreddit':       None,
    'post_title':      None,
    'post_score':      None,
    'comment_score':   None,
    'rating':          df_kaggle['score'],
    'thumbs_up':       df_kaggle['thumbsUpCount'],
    'vader_compound':  vader_scores['vader_compound'].values,
    'vader_positive':  vader_scores['vader_positive'].values,
    'vader_negative':  vader_scores['vader_negative'].values,
    'vader_neutral':   vader_scores['vader_neutral'].values,
    'sentiment_label': vader_scores['sentiment_label'].values,
    'category_1':      None,
    'category_2':      None,
    'category_3':      None,
    'dimension_1':     None,
    'dimension_2':     None,
})

print(f'Kaggle standardised: {len(df_kaggle_std)} rows')
print(f'Date range: {df_kaggle_std["original_date"].min()} to {df_kaggle_std["original_date"].max()}')

In [ ]:
import os
print(os.listdir('.'))

In [ ]:
# ---------------------------------------------------------------
# MERGE AND DEDUPLICATE
# ---------------------------------------------------------------

# Align columns before concat
df_combined   = df_combined.reindex(columns=FINAL_COLUMNS)
df_kaggle_std = df_kaggle_std.reindex(columns=FINAL_COLUMNS)

# Merge
df_full = pd.concat([df_combined, df_kaggle_std], ignore_index=True)

# Standardise date
df_full['original_date'] = pd.to_datetime(df_full['original_date'], errors='coerce')
df_full['comment_date']  = pd.to_datetime(df_full['comment_date'],  errors='coerce')

# Deduplicate on text + source in case of overlap
before_dedup = len(df_full)
df_full = df_full.drop_duplicates(subset=['text', 'source']).reset_index(drop=True)
after_dedup  = len(df_full)

print(f'Rows before dedup: {before_dedup}')
print(f'Rows after dedup:  {after_dedup}')
print(f'Duplicates removed: {before_dedup - after_dedup}')

# Sort by date
df_full = df_full.sort_values('original_date', ascending=False).reset_index(drop=True)

print(f'\nFinal combined dataset: {len(df_full)} rows')

In [ ]:
# ---------------------------------------------------------------
# SUMMARY
# ---------------------------------------------------------------
print('COMBINED FULL DATASET SUMMARY')
print('=' * 60)
print(f'Total rows: {len(df_full)}')
print(f'\nBy source:')
print(df_full['source'].value_counts().to_string())
print(f'\nDate range: {df_full["original_date"].min().date()} to {df_full["original_date"].max().date()}')
print(f'\nSentiment breakdown:')
print(df_full['sentiment_label'].value_counts(normalize=True).mul(100).round(1).to_string())
print(f'\nAvg star rating (app store sources):')
print(df_full.groupby('source')['rating'].mean().round(2).to_string())
print(f'\nQuarterly sentiment trend:')
df_full['quarter'] = df_full['original_date'].dt.to_period('Q')
print(df_full.groupby('quarter')['vader_compound'].mean().round(3).to_string())
df_full = df_full.drop(columns=['quarter'])

In [ ]:
df_full.to_csv(OUTPUT_FILE, index=False)
print(f'Saved {len(df_full)} rows to {OUTPUT_FILE}')
print(f'Columns ({len(df_full.columns)}): {list(df_full.columns)}')

In [ ]:
try:
    from google.colab import files
    files.download(OUTPUT_FILE)
    print('Download triggered.')
except ImportError:
    print(f'Not in Colab — file saved locally as {OUTPUT_FILE}')